In [ ]:
library(dplyr)
library(lubridate)

# --- IF YOU HAVE THE .RData FILES IN COLAB, UNCOMMENT THESE TWO LINES: ---
# load("congress.RData")
# load("senator_tweets.RData")

# --- SELF-CONTAINED IN-MEMORY DATASETS (No external files needed) ---
set.seed(101)

# 1. Congress members dataset (Senators + Representatives, with some missing Twitter handles)
congress <- data.frame(
  name = c(
    "Sen_Smith", "Sen_Johnson", "Sen_Williams", "Sen_Brown", "Sen_Jones",
    "Sen_Garcia", "Sen_Miller", "Sen_Davis", "Sen_Wilson", "Sen_Taylor",
    "Rep_Moore", "Rep_Anderson"
  ),
  chamber = c(rep("Senate", 10), rep("House", 2)),
  gender = c("F", "M", "F", "M", "F", "M", "M", "F", "M", "F", "M", "F"),
  birthyear = c(1958, 1965, 1972, 1952, 1968, 1961, 1975, 1964, 1955, 1970, 1980, 1978),
  twitter = c(
    "SenSmith", "SenJohnson", "SenWilliams", "SenBrown", NA,
    "SenGarcia", "SenMiller", "SenDavis", NA, "SenTaylor",
    "RepMoore", "RepAnderson"
  ),
  stringsAsFactors = FALSE
)

# 2. Twitter user metadata returned by API (7 of the 8 listed handles found on Twitter)
senator_users <- data.frame(
  screen_name = c(
    "SenSmith", "SenJohnson", "SenWilliams", "SenBrown",
    "SenGarcia", "SenMiller", "SenDavis" # Note: SenTaylor was not found / suspended
  ),
  statuses_count = c(14200, 9850, 18400, 7600, 11300, 15900, 12750),
  account_created_at = as.POSIXct(c(
    "2009-03-15 10:20:00", "2011-06-22 14:05:00", "2013-01-10 09:30:00",
    "2009-11-04 16:45:00", "2012-08-19 11:15:00", "2014-05-02 08:50:00",
    "2010-09-28 13:40:00"
  )),
  stringsAsFactors = FALSE
)

# 3. Timeline tweets dataset (senator_tweets: multiple tweets per Senator with timestamps & retweets)
senator_tweets <- expand.grid(
  screen_name = senator_users$screen_name,
  tweet_num = 1:5,
  stringsAsFactors = FALSE
) %>%
  mutate(
    created_at = as.POSIXct("2026-11-01 12:00:00") - (tweet_num * 86400) - sample(100:5000, n(), replace = TRUE),
    text = paste("Legislative update #", tweet_num, "from", screen_name),
    retweet_count = sample(40:650, n(), replace = TRUE)
  )

In [ ]:
# Filter for Senators only
senators <- congress %>%
  filter(chamber == "Senate")

# Senators with a Twitter handle listed in the congress dataset
listed_senators <- senators %>%
  filter(!is.na(twitter) & twitter != "")

# Match listed handles against the accounts actually found via the Twitter API
found_senators <- listed_senators %>%
  inner_join(senator_users, by = c("twitter" = "screen_name"))

cat("Total Senators in dataset:", nrow(senators), "\n")
cat("Senators with listed Twitter handles:", nrow(listed_senators), "\n")
cat("Senator Twitter accounts found on Twitter:", nrow(found_senators), "\n")

In [ ]:
avg_tweets_by_gender <- found_senators %>%
  group_by(gender) %>%
  summarise(
    senator_count = n(),
    avg_published_tweets = mean(statuses_count, na.rm = TRUE)
  )

print(avg_tweets_by_gender)

In [ ]:
# Calculate Senator age and convert account creation timestamp to numeric
senator_age_df <- found_senators %>%
  mutate(
    age = 2026 - birthyear,
    created_numeric = as.numeric(account_created_at)
  )

# Compute correlation
correlation_val <- cor(senator_age_df$age, senator_age_df$created_numeric, use = "complete.obs")
cat("Correlation between Senator age and account creation time:", round(correlation_val, 4), "\n\n")

# Detailed statistical significance test
cor.test(senator_age_df$age, senator_age_df$created_numeric)

In [ ]:
# 1. Collect the last 2 tweets published by each Senator
last_two_tweets <- senator_tweets %>%
  arrange(screen_name, desc(created_at)) %>%
  group_by(screen_name) %>%
  slice_head(n = 2) %>%
  ungroup()

# 2. Join with Senator gender and compute average retweets by gender
avg_retweets_by_gender <- last_two_tweets %>%
  inner_join(senators %>% select(name, gender, twitter), by = c("screen_name" = "twitter")) %>%
  group_by(gender) %>%
  summarise(
    total_tweets_sampled = n(),
    avg_retweets = mean(retweet_count, na.rm = TRUE)
  )

print(avg_retweets_by_gender)